In [12]:
import rasterio as rs
import numpy as np
import matplotlib.pyplot as plt

import geopandas as gpd
import shapely
import fiona
import pandas as pd
import json
import os
from glob import glob
from shapely.geometry import Polygon

from sklearn.metrics import f1_score
from sarpy.visualization.remap import density
from rasterio.windows import from_bounds
from shapely.geometry import box

from sarpy.visualization.remap import density, brighter

import geopandas as gpd
import fiona
from tqdm import tqdm
from shapely import make_valid
from datetime import datetime
from datetime import timedelta

from rasterio.windows import Window

In [ ]:
fiona.supported_drivers['KML'] = 'rw'

all_kml_files = glob('/data/datasets/olmo/usfs/raw/Annotations/*/*.kml')

gdf=None
for i, f in enumerate(tqdm(all_kml_files)):
    
    metadata = json.load(open('/'.join(f.split('/')[:8]) + '/metadata.json', 'r'))

    gdf_ = gpd.read_file(f, driver='KML')
    gdf_['cluster_id'] = i
    start_time = datetime(year=int(metadata['Year']), month=int(metadata['Month']), day=1)
    end_time = start_time + timedelta(days=30)
    
    gdf_['start_time'] = start_time
    gdf_['end_time'] = end_time
    
    gdf_.geometry = gdf_.geometry.apply(lambda g : make_valid(g))
    
    if gdf is None:
        gdf = gdf_.copy()
    else:
        gdf = pd.concat([gdf, gdf_], axis=0, ignore_index=True)
        
    
# disolve by cluster ID
gdf.to_crs(32612, inplace=True)
gdf['geom_area'] = gdf.geometry.area
clusters = gdf.dissolve(by="cluster_id")
clusters.to_crs(32612, inplace=True)

In [14]:
gdf.shape, clusters.shape

((627, 19), (23, 18))

# Define task geom

In [15]:
from shapely import is_valid, is_valid_reason, make_valid

In [16]:
clusters['task_geom'] = clusters.geometry.apply(lambda r: r.oriented_envelope.envelope)

In [17]:
clusters['valid_geometry'] = clusters['geometry'].apply(lambda r : is_valid(r))
clusters.geometry = clusters['geometry'].apply(lambda r : make_valid(r))
clusters['valid_geometry'] = clusters['geometry'].apply(lambda r : is_valid(r))

In [18]:
task_geom_bounds = clusters['task_geom'].bounds
clusters['task_geom_width'] = task_geom_bounds.maxx - task_geom_bounds.minx
clusters['task_geom_height'] = task_geom_bounds.maxy - task_geom_bounds.miny

In [19]:
MIN_HW_METERS = 64 * 3 #64pix * 3m/pix -->> this is based on patch sized used in Olmo (64 pix in config)
EPS = 10 # buffer in meters

# find min sude
clusters['min_side'] = np.minimum(clusters.task_geom_width.values, clusters.task_geom_height.values)

# buffer by min side diff
clusters['min_buffer'] = np.maximum(0, MIN_HW_METERS - clusters['min_side'] + EPS)

# apply buffer to task_geom - this new column will be used in the OER processing python script
clusters['task_geom_buff'] = clusters.apply(lambda r: r['task_geom'].buffer(r.min_buffer, 
                                                                            cap_style='square', 
                                                                            join_style='mitre'), 
                                                    axis=1)

# define labels

In [20]:
clusters['label'] = 'infected'

# save out

In [21]:
tmp = gpd.GeoSeries(clusters["task_geom_buff"], crs=clusters.crs)
task_bbox_wgs84 = tmp.to_crs(4326).envelope
clusters['task_geom_buff'] = task_bbox_wgs84.to_wkt()

tmp = gpd.GeoSeries(clusters["task_geom"], crs=clusters.crs)
task_bbox_wgs84 = tmp.to_crs(4326).envelope
clusters['task_geom'] = task_bbox_wgs84.to_wkt()

#convert EPSG
clusters.to_crs('EPSG:4326', inplace=True)

# save geopandas dataframe
clusters.to_file("usfs/raw/cluster_all.gdb", driver="GPKG")

# edit rasterized labels for background clutter

In [ ]:
label_files = glob(f'oerun/dataset/windows/spatial_split_10km/task_*/layers/label/label/geotiff.tif')
len(label_files)

In [ ]:
for orig_file in tqdm(label_files):
    
    img_rs_orig = rs.open(orig_file)
    img_orig = img_rs_orig.read(1)

    ## **** overwrite here
    img_orig[img_orig==0] = 1
    img_orig[img_orig==255] = 0

    # break
    with rs.open(orig_file, "w", **img_rs_orig.profile) as dst:
        dst.write(img_orig, 1)

In [ ]:
plt.figure(figsize=(20,20))
plt.imshow(np.ma.array(img_orig, mask=img_orig==255))
plt.colorbar(shrink=0.25)
plt.show()